# BT4221 Project
This file contains the EDA findings for our BT4221 Project. We will approach this file in the following order:
1. Imports and setup
2. Data Preparation
3. Duplicate Data Analysis
4. Missing Data Analysis
5. Sampling from Uncorrupted Raw Data
6. Univariate Analysis
7. Bivariate Analysis
8. Covariance Matrix
9. Text Analysis

At the end of this file, our goal is to obtain a deeper understanding of our data, knowing what we are working with, as well as knowing what are the appropriate steps to take for feature selection and engineering. Additionally, the dataset obtained from sampling contains our fully cleaned dataset.

Member names and student numbers:

1. Josh Lim, A0251991X
2. 
3. 
4. 

## Imports and setup
We will first begin with preparing our environment. This will involve all admin imports, spark setups, and helper functions

### Import Packages
We will import all packages that will be used in the notebook file here.

In [1]:
import kagglehub
from pyspark.sql import SparkSession
import os
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
from pyspark.sql.functions import (
    col, when, count, mean, stddev,
    min as spark_min, max as spark_max,
    percentile_approx, trim, lower, rank,
    isnan, length, size, split,
    expr as spark_expr
)
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.window import Window
from pyspark import StorageLevel
from scipy.stats import chi2_contingency

/Users/joshlim/Desktop/y4s2/bt4221/BT4221-Project-Steam-Reviews/bt4221projvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Get Dataset Path
We will obtain the dataset from kaggle and assign it a dataset path.

In [2]:
# Download latest version
path = kagglehub.dataset_download("najzeko/steam-reviews-2021")

print("Path to dataset files:", path)

Path to dataset files: /Users/joshlim/.cache/kagglehub/datasets/najzeko/steam-reviews-2021/versions/1


### Build SparkSession
We will set up the spark session, as well as set the log level to error to hide certain noisy outputs.

In [3]:
spark = SparkSession.builder.appName("BT4221_Project").getOrCreate()
print("Spark version:", spark.version)
# Suppress the CSV header warning

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 00:06:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/09 00:06:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


### Helper Functions
We will set up all required helper functions here to make the notebook neater.

In [4]:
OUTPUT_DIR = "eda_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def show(fig, filename):
    filepath = os.path.join(OUTPUT_DIR, f"{filename}.html")
    fig.write_html(filepath)
    print(f"Plot saved → {filepath}")

## Data Preparation
We will begin with some simple data preparation steps that fix most of the surface level issues with our data. This is a key step to take before we do any analysis to ensure that our analysis is not biased due to certain corruptions in the data. 

### Load data
We will first begin by loading the data into df.

In [5]:
# Init spark session
spark = SparkSession.builder \
    .appName("SteamReviews") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# load csv into df
df = spark.read.csv(
    os.path.join(path, "steam_reviews.csv"),
    header=True,
    inferSchema=False,
    multiLine=True,
    escape='"'
)

### Drop Index Column
There exists an index column, in which we will drop as it serves no purpose when it comes to our analysis as well as our modelling.

In [6]:
# Drop index column
df = df.drop("_c0")

### Fix Column Names
We will want to follow proper naming conventions, and hence we want to exclude dots from the names of our features. We will instead replace them with underscores.

In [7]:
# Replace . with _ in names
for c in df.columns:
    if "author." in c:
        df = df.withColumnRenamed(c, c.replace("author.", "author_"))

### Standardise Data Types
We will next move on to standardising the datatypes of our features. For boolean features, we want to change them to binary. For Numerics, a double is suitable. Lastly, timestamps will be casted to bigint. We also want to lower and strip the values in our categorical features, app_name and language.

In [8]:
# Boolean to int
bool_cols = ["recommended", "steam_purchase", "received_for_free", "written_during_early_access"]
for c in bool_cols:
    df = df.withColumn(c, when(col(c) == "True", 1).otherwise(0).cast("int"))

# Numeric to double
num_cols = [
    "votes_helpful", "votes_funny", "weighted_vote_score", "comment_count",
    "author_num_games_owned", "author_num_reviews", "author_playtime_forever",
    "author_playtime_last_two_weeks", "author_playtime_at_review", "author_last_played"
]
for c in num_cols:
    df = df.withColumn(c, spark_expr(f"try_cast(`{c}` AS DOUBLE)"))

# Timestamps to bigint
for c in ["timestamp_created", "timestamp_updated"]:
    df = df.withColumn(c, spark_expr(f"try_cast(try_cast(`{c}` AS DOUBLE) AS BIGINT)"))

# Standardise categorical columns
df = df.withColumn("language", trim(lower(col("language"))))
df = df.withColumn("app_name", trim(col("app_name")))

### Filter for English Rows
We will filter for reviews that are in english to keep, and discard the rest. This is because we are facing a limitation where translation is extremely computationally expensive, and hence we will be using only english data.

In [9]:
# Remove non english rows
df = df.filter(col("language") == "english")

### Verify Changes
We will print the Schema again and a few sample rows to verify that the changes that we made worked properly, before we move on to performing any analysis.

In [10]:
# Print to verify changes
print(f"Total rows: {df.count():,}")
print("\nSchema:")
df.printSchema()
print("\nSample rows:")
df.show(3)

Total rows: 9,635,437

Schema:
root
 |-- app_id: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- language: string (nullable = true)
 |-- review: string (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- recommended: integer (nullable = false)
 |-- votes_helpful: double (nullable = true)
 |-- votes_funny: double (nullable = true)
 |-- weighted_vote_score: double (nullable = true)
 |-- comment_count: double (nullable = true)
 |-- steam_purchase: integer (nullable = false)
 |-- received_for_free: integer (nullable = false)
 |-- written_during_early_access: integer (nullable = false)
 |-- author_steamid: string (nullable = true)
 |-- author_num_games_owned: double (nullable = true)
 |-- author_num_reviews: double (nullable = true)
 |-- author_playtime_forever: double (nullable = true)
 |-- author_playtime_last_two_weeks: double (nullable = true)
 |-- author_playt

## Duplicate Data Analysis
We want to identify any duplicate data within our dataset and handle them appropriately. For our case specifically, there are 3 types of duplicate data:

1. Exact Duplicates
2. Duplicate Review ID
3. Duplicate user ID and app name

In the case of 1, it is safe to completely drop all duplicates, to remove any potential bias.
In the case of 2, it is also safe to completely drop duplicates, since they are supposed to be the same review.
In the case of 3, we will only keep the most recent review, as it could be a re-review of the same game by the same author.

### Find Exact Duplicates
We will start by identifying exact duplicates.

In [11]:
total = df.count()

exact_dupes = total - df.dropDuplicates().count()
print(f"Exact duplicate rows: {exact_dupes:,} ({exact_dupes/total*100:.4f}%)")

Exact duplicate rows: 54,769 (0.5684%)


### Identify Duplicate Review IDs
We will now move on to identifying duplicate review IDs. 

In [12]:
dupe_review_ids = total - df.dropDuplicates(["review_id"]).count()
print(f"Duplicate review_ids: {dupe_review_ids:,} ({dupe_review_ids/total*100:.4f}%)")

if dupe_review_ids > 0:
    print("\nSample duplicate review_ids:")
    df.groupBy("review_id") \
        .count() \
        .filter(col("count") > 1) \
        .orderBy(col("count").desc()) \
        .limit(10) \
        .show(truncate=False)

Duplicate review_ids: 54,769 (0.5684%)

Sample duplicate review_ids:


+---------+-----+
|review_id|count|
+---------+-----+
|30152063 |2    |
|30150973 |2    |
|30154181 |2    |
|30152283 |2    |
|30153540 |2    |
|30153136 |2    |
|30155078 |2    |
|30153807 |2    |
|30151713 |2    |
|30156798 |2    |
+---------+-----+



### Identify Re-Reviews
We will now move on to identify re-reviews.

In [13]:
dupe_user_app = total - df.dropDuplicates(["author_steamid", "app_id"]).count()
print(f"Same user + same app: {dupe_user_app:,} ({dupe_user_app/total*100:.4f}%)")

if dupe_user_app > 0:
    print("\nSample users who reviewed same app multiple times:")
    df.groupBy("author_steamid", "app_id") \
        .count() \
        .filter(col("count") > 1) \
        .orderBy(col("count").desc()) \
        .limit(10) \
        .show(truncate=False)

    # Check if these are review edits (different timestamps)
    print("\nNumber of Unique Timestamps")
    df.groupBy("author_steamid", "app_id") \
        .agg(
            count("*").alias("review_count"),
            count(col("timestamp_updated").cast("string")).alias("unique_timestamps")
        ) \
        .filter(col("review_count") > 1) \
        .orderBy(col("review_count").desc()) \
        .limit(10) \
        .show(truncate=False)

Same user + same app: 55,027 (0.5711%)

Sample users who reviewed same app multiple times:


+-----------------+------+-----+
|author_steamid   |app_id|count|
+-----------------+------+-----+
|76561198022657933|250900|16   |
|76561198146250120|239030|12   |
|76561198140979823|206190|8    |
|76561198095702682|206440|8    |
|76561198098752384|221380|8    |
|76561198035773921|346110|7    |
|76561198063133824|107410|4    |
|76561198061858276|346110|4    |
|76561198105610448|206190|4    |
|76561197987190386|72850 |4    |
+-----------------+------+-----+


Number of Unique Timestamps


+-----------------+------+------------+-----------------+
|author_steamid   |app_id|review_count|unique_timestamps|
+-----------------+------+------------+-----------------+
|76561198022657933|250900|16          |16               |
|76561198146250120|239030|12          |12               |
|76561198095702682|206440|8           |8                |
|76561198140979823|206190|8           |8                |
|76561198098752384|221380|8           |8                |
|76561198035773921|346110|7           |7                |
|76561197969422751|242920|4           |4                |
|76561198030980450|214950|4           |4                |
|76561198064129864|113200|4           |4                |
|76561198033608126|213670|4           |4                |
+-----------------+------+------------+-----------------+



### Handle Duplicates
After identifying the duplicates, we will handle each one accordingly.

In [14]:
# For same user + same app, keep the most recently updated review
window = Window.partitionBy("author_steamid", "app_id").orderBy(col("timestamp_updated").desc())
df = df.withColumn("rank", rank().over(window)) \
       .filter(col("rank") == 1) \
       .drop("rank")

# Drop any remaining exact duplicates
df = df.dropDuplicates()

final = df.count()
print(f"Rows before: {total:,}")
print(f"Rows after:  {final:,}")
print(f"Rows dropped: {total - final:,} ({(total - final) / total * 100:.4f}%)")

Rows before: 9,635,437
Rows after:  9,580,551
Rows dropped: 54,886 (0.5696%)


## Missing Data Analysis
It is important that we handle missing data appropriately as well. Missing data comes in 3 forms:

1. MCAR: When data is missing completely at random, it is safe to drop and move on.
2. MAR: When data is missing at random, it is usually also safe to drop, however more precaution should be taken.
3. MNAR: When data is not missing at random, it is important that we identify what is causing the data to be missing, and not to drop them as we are potentially losing valuable information.

In [15]:
dtype_map = dict(df.dtypes)

missing_counts = df.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
    if dtype_map[c] in ("double", "float")
    else count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0]

missing_df = pd.DataFrame([{
    "column":        c,
    "missing_count": missing_counts[c],
    "missing_pct":   round(missing_counts[c] / total * 100, 4),
    "dtype":         dtype_map[c]
} for c in df.columns]).sort_values("missing_pct", ascending=False)

print(missing_df.to_string(index=False))

                        column  missing_count  missing_pct  dtype
                        review          16678       0.1731 string
     author_playtime_at_review          11836       0.1228 double
                        app_id              0       0.0000 string
                steam_purchase              0       0.0000    int
author_playtime_last_two_weeks              2       0.0000 double
       author_playtime_forever              2       0.0000 double
            author_num_reviews              0       0.0000 double
        author_num_games_owned              0       0.0000 double
                author_steamid              0       0.0000 string
   written_during_early_access              0       0.0000    int
             received_for_free              0       0.0000    int
                 comment_count              0       0.0000 double
                      app_name              0       0.0000 string
           weighted_vote_score              0       0.0000 double
          

### Drop Missing Rows
We do not need to do missing data analysis, as we can see that only a small portion of our data is missing, and hence the effect of dropping all missing rows is negligible. 

In [16]:
before = df.count()
df = df.dropna()
after = df.count()

print(f"Rows before: {before:,}")
print(f"Rows after:  {after:,}")
print(f"Rows dropped: {before - after:,} ({(before - after) / before * 100:.4f}%)")

Rows before: 9,580,551
Rows after:  9,552,039
Rows dropped: 28,512 (0.2976%)


## Sample From Uncorrupted Raw Data
We will now sample our data to occupy roughly 500mb of memory. This is done after removing duplicate, missing, and currupted rows in general bacause we do not want our sampled data to contain any of these rows. However, other data cleaning and preparation steps, as well as feature selection and feature engineering, will be done on the sampled dataset. This is because we want to just ensure that the 500mb of data that we have is legitimate and valid data. Other steps that are done are purely just choosing the best datapoints and features for maximising our modelling ability.

### Obtain Individual Class count
We will first identify the ratio between the classes of our target variable. We want to end up with a balanced dataset with equal representation of both classes, hence this ratio will be useful.

In [17]:
# Get class count
class_counts = df.groupBy("recommended").count().collect()
count_dict = {row["recommended"]: row["count"] for row in class_counts}
print(f"Class counts: {count_dict}")

Class counts: {1: 8505522, 0: 1046517}


### Compute Balancing Fractions
We will compute the fraction required to balance the classes. This is so that we take into account the disparity between the majority and minority classes, while trying to sample our dataset to a 500mb size.

In [23]:
# Minority class count
minority_count = min(count_dict.values())

# Sampling fraction
fractions = {
    cls: min((minority_count / count), 1.0)
    for cls, count in count_dict.items()
}
print(f"Sampling fractions: {fractions}")

Sampling fractions: {1: 0.12303971467006963, 0: 1.0}


### Stratified Random Sampling
We will then use the sampling fraction to perform stratified random sampling on our dataset. Stratified random sampling is done to ensure that we end up with a balanced dataset, where both reccommended and not reccommended are both equally represented. It is also random to ensure that the sampled dataset does not contain any selection bias or any other factors that might affect our model performance.

In [19]:
# Stratified sample
df_sample = df.sampleBy("recommended", fractions=fractions, seed=42)

# Verfy
sample_count = df_sample.count()
print(f"\nSample rows: {sample_count:,}")
print("\nClass balance in sample:")
df_sample.groupBy("recommended").count() \
    .withColumn("pct", col("count") / sample_count * 100) \
    .orderBy("recommended") \
    .show()


Sample rows: 696,134

Class balance in sample:


+-----------+------+-----------------+
|recommended| count|              pct|
+-----------+------+-----------------+
|          0|348432| 50.0524324339854|
|          1|347702|49.94756756601459|
+-----------+------+-----------------+



### Caching Sampled Data
We will then proceed to cache the sampled data, and start performing analysis using this dataset. This is to save us on compute time. Additionally, the dataset is cached now because we will not be making any changes to the dataset itself until we finish with the exploratory data analysis, in which we will then make neccessary changes if required, before moving on to our feature selection and engineering.

In [20]:
# Caching (difficulty running, do not attempt)



# df = df_sample

# df.persist(StorageLevel.MEMORY_AND_DISK)
# sample_count = df.count()  # triggers cache

# # Verify it's cached
# for rdd in spark.sparkContext._jsc.sc().getRDDStorageInfo():
#     mem_gb  = rdd.memSize() / 1024**3
#     disk_gb = rdd.diskSize() / 1024**3
#     print(f"Memory used: {mem_gb:.2f} GB")
#     print(f"Disk used:   {disk_gb:.2f} GB")

# print(f"\nSample rows: {sample_count:,}")
# print("Sample cached and ready for EDA.")

## Univariate Analysis
We will do some univariate analysis to try and uncover any potentially useful information. We are looking out for potential outliers to drop, or potential inconsistencies that require our attention.

### Numeric Univariate Analysis
We will plot the distribution of each numeric feature, for us to observe if there is any obvious outlier, and whether or not our data follows a gaussian curve. 

In [21]:
# Sample to pandas for plotting
sample_pd = df_sample.select(num_cols).toPandas()

for c in num_cols:
    fig = px.histogram(
        sample_pd,
        x=c,
        nbins=100,
        title=f"Distribution of {c}",
        labels={c: c, "count": "Frequency"},
        marginal="box"
    )
    fig.update_layout(bargap=0.1)
    show(fig, f"phase4_univariate_{c}")
    print(f"Saved distribution plot for {c}")

Plot saved → eda_plots/phase4_univariate_votes_helpful.html
Saved distribution plot for votes_helpful
Plot saved → eda_plots/phase4_univariate_votes_funny.html
Saved distribution plot for votes_funny
Plot saved → eda_plots/phase4_univariate_weighted_vote_score.html
Saved distribution plot for weighted_vote_score
Plot saved → eda_plots/phase4_univariate_comment_count.html
Saved distribution plot for comment_count
Plot saved → eda_plots/phase4_univariate_author_num_games_owned.html
Saved distribution plot for author_num_games_owned
Plot saved → eda_plots/phase4_univariate_author_num_reviews.html
Saved distribution plot for author_num_reviews
Plot saved → eda_plots/phase4_univariate_author_playtime_forever.html
Saved distribution plot for author_playtime_forever
Plot saved → eda_plots/phase4_univariate_author_playtime_last_two_weeks.html
Saved distribution plot for author_playtime_last_two_weeks
Plot saved → eda_plots/phase4_univariate_author_playtime_at_review.html
Saved distribution plo

There are some interesting observations from the univariate distribution plots. Every single plot contained a decent amount of outliers, with a large majority of the data being centered in low values, with some rows having extremely large values. However, based on domain knowledge, such cases are actually possible. Hence, the longtail distribution does not actually contain any faulty or errornous values. As such, all outliers identified here will be left as is, due to supporting domain knowledge.

For votes_helpful, votes_funny, comment_count, and weighted_vote_score, certain comments get a large amount of traction, which cause them to recieve large amounts of votes and comments.

For author related statistics, there are certain authors who own a large number of games, and have a generally large amount of playtime, which would make perfect logical sense. Majority of users are not diehard gamers, but there are a small handful who are.

### Categorical Univariate Analysis
We will then plot the distribution of the categorical features as well, to again identify any potential information.

In [22]:
# High cardinality categoricals
cat_cols = ["language", "app_name"]

for c in cat_cols:
    # Aggregate in Spark, then plot
    cat_df = df_sample.groupBy(c) \
        .count() \
        .orderBy(col("count").desc()) \
        .limit(20) \
        .toPandas()

    fig = px.bar(
        cat_df,
        x=c, y="count",
        title=f"Top 20 values — {c}",
        labels={"count": "Number of reviews", c: c},
        text="count",
        color="count",
        color_continuous_scale="Blues"
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    fig.update_layout(xaxis_tickangle=-45)
    show(fig, f"phase4_univariate_{c}")
    print(f"Saved plot for {c}")

# Binary Categoricals
binary_cols = ["recommended", "steam_purchase", "received_for_free", "written_during_early_access"]

for c in binary_cols:
    bin_df = df_sample.groupBy(c) \
        .count() \
        .withColumn("pct", col("count") / sample_count * 100) \
        .orderBy(c) \
        .toPandas()
    bin_df[c] = bin_df[c].map({0: "No", 1: "Yes"})

    fig = px.pie(
        bin_df,
        names=c, values="count",
        title=f"Distribution of {c}",
        hole=0.4,   # donut chart — easier to read proportions than a full pie
        color_discrete_sequence=["#636EFA", "#EF553B"]
    )
    fig.update_traces(texttemplate="%{label}: %{percent:.1%}")
    show(fig, f"phase4_univariate_{c}")
    print(f"Saved plot for {c}")

Plot saved → eda_plots/phase4_univariate_language.html
Saved plot for language


Plot saved → eda_plots/phase4_univariate_app_name.html
Saved plot for app_name


Plot saved → eda_plots/phase4_univariate_recommended.html
Saved plot for recommended


Plot saved → eda_plots/phase4_univariate_steam_purchase.html
Saved plot for steam_purchase


Plot saved → eda_plots/phase4_univariate_received_for_free.html
Saved plot for received_for_free


Plot saved → eda_plots/phase4_univariate_written_during_early_access.html
Saved plot for written_during_early_access


For our high cardinality categoricals, we are checking language just to verify that only english reviews remain. Next, looking at app_name, we do see a very good range of counts across each name, with no singular class dominating. This is a healthy sign that the distribution for app_name is decently balanced.

What was more worrying was that among our binary categoricals, written_during_early_access, steam_purchase, and received_for_free all had a strong class imbalance. Though it makes sense that these are potentially rare categories, this is insightful information and we have to look into the bivariate analysis to understand the usefulness of these features. If they are predictive, we might want to ensure that we have good class representation for them.

## Bivariate Analysis
Now we will be comparing each individual feature to the target variable, which allows us to identify if there are any features with strong predictive powers that we should take note of, or vice versa. This is important especially with feature selection and engineering.

### Numeric Bivariate Analysis
We will try by visually identifying if there is any difference in the distribution of each numeric feature between classes. If differences are present, it tells us that the feature is potentially predictive, and vice versa. 

In [23]:
# Convert to pandas for plotting
sample_pd = df_sample.select(num_cols + ["recommended"]).toPandas()
sample_pd["recommended"] = sample_pd["recommended"].map({0: "Not Recommended", 1: "Recommended"})

for c in num_cols:
    # Box plot
    fig = px.box(
        sample_pd,
        x="recommended", y=c,
        title=f"{c} by recommended",
        labels={"recommended": "Recommended", c: c},
        color="recommended",
        color_discrete_map={
            "Not Recommended": "#EF553B",
            "Recommended":     "#636EFA"
        }
    )
    show(fig, f"phase5_bivariate_box_{c}")

    # Overlapping histogram
    fig = px.histogram(
        sample_pd,
        x=c,
        color="recommended",
        nbins=100,
        barmode="overlay",
        opacity=0.6,
        title=f"Distribution of {c} by recommended",
        labels={"recommended": "Recommended", c: c},
        color_discrete_map={
            "Not Recommended": "#EF553B",
            "Recommended":     "#636EFA"
        }
    )
    show(fig, f"phase5_bivariate_hist_{c}")
    print(f"Saved plots for {c}")

# Mean comparisons
print("\n=== Mean of numeric features by recommended ===")
df_sample.groupBy("recommended") \
    .agg(*[mean(c).alias(f"mean_{c}") for c in num_cols]) \
    .orderBy("recommended") \
    .show(truncate=False)

Plot saved → eda_plots/phase5_bivariate_box_votes_helpful.html
Plot saved → eda_plots/phase5_bivariate_hist_votes_helpful.html
Saved plots for votes_helpful
Plot saved → eda_plots/phase5_bivariate_box_votes_funny.html
Plot saved → eda_plots/phase5_bivariate_hist_votes_funny.html
Saved plots for votes_funny
Plot saved → eda_plots/phase5_bivariate_box_weighted_vote_score.html
Plot saved → eda_plots/phase5_bivariate_hist_weighted_vote_score.html
Saved plots for weighted_vote_score
Plot saved → eda_plots/phase5_bivariate_box_comment_count.html
Plot saved → eda_plots/phase5_bivariate_hist_comment_count.html
Saved plots for comment_count
Plot saved → eda_plots/phase5_bivariate_box_author_num_games_owned.html
Plot saved → eda_plots/phase5_bivariate_hist_author_num_games_owned.html
Saved plots for author_num_games_owned
Plot saved → eda_plots/phase5_bivariate_box_author_num_reviews.html
Plot saved → eda_plots/phase5_bivariate_hist_author_num_reviews.html
Saved plots for author_num_reviews
Plot

+-----------+------------------+-----------------+------------------------+------------------+---------------------------+-----------------------+----------------------------+-----------------------------------+------------------------------+-----------------------+
|recommended|mean_votes_helpful|mean_votes_funny |mean_weighted_vote_score|mean_comment_count|mean_author_num_games_owned|mean_author_num_reviews|mean_author_playtime_forever|mean_author_playtime_last_two_weeks|mean_author_playtime_at_review|mean_author_last_played|
+-----------+------------------+-----------------+------------------------+------------------+---------------------------+-----------------------+----------------------------+-----------------------------------+------------------------------+-----------------------+
|0          |6.631322611011618 |406777.8219882215|0.33784737929170444     |0.5525238784038206|207.60943024750884         |15.935620149699224     |15182.559739059558          |92.58129850300776       

From here, we are mainly looking at whether there is a difference in the mean of each feature across reccommended to determine if there is a strong differentiation within each feature. We can see that for almost all features, there is at least a certain level of difference in the mean values. The next thing we need to look at is the difference in the variance, primarily by looking at the boxplots. There is generally also a significant difference in the IQR that we can take note of, showing that most of these features are predictive. The exception being mean_author_last_played, which is showing relatively similar mean values.

### Categorical Bivariate Analysis
We will perform a similar analysis as for numerical, however, since the data is categorical, we will use a different approach as we do not have access to statistics such as median, mean, IQR, etc etc. For categorical, we will use a reccommendation rate calculation, as well as a chi-squared test to see the predictivity of a feature.

In [24]:
# app_name vs reccommended
print("\n=== App Name vs Recommended ===")

app_rec = df_sample.groupBy("app_name") \
    .agg(
        count("*").alias("total"),
        mean("recommended").alias("rec_rate")
    ) \
    .withColumn("rec_rate_pct", col("rec_rate") * 100) \
    .orderBy(col("total").desc()) \
    .limit(20) \
    .toPandas()

print(app_rec.to_string(index=False))

# Top 20 apps by review count — recommendation rate
fig = px.bar(
    app_rec,
    x="app_name", y="rec_rate_pct",
    title="Recommendation rate by app (top 20 most reviewed, %)",
    labels={"rec_rate_pct": "Recommendation rate (%)", "app_name": "App"},
    text="rec_rate_pct",
    color="rec_rate_pct",
    color_continuous_scale="RdYlGn"
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(xaxis_tickangle=-45)
show(fig, "phase5_bivariate_app_rec_rate")

# Chi-Square test
print("\n=== Chi-square test ===")

for c in ["app_name"]:
    # Build contingency table
    contingency = df_sample.groupBy(c, "recommended") \
        .count() \
        .toPandas() \
        .pivot(index=c, columns="recommended", values="count") \
        .fillna(0)

    chi2, pval, dof, expected = chi2_contingency(contingency)
    print(f"\n{c}:")
    print(f"  Chi2 statistic: {chi2:.4f}")
    print(f"  p-value:        {pval:.6f}")
    print(f"  Degrees of freedom: {dof}")
    print(f"  Significant (p < 0.05): {pval < 0.05}")


=== App Name vs Recommended ===


                                   app_name  total  rec_rate  rec_rate_pct
              PLAYERUNKNOWN'S BATTLEGROUNDS  49090  0.163333     16.333265
                         Grand Theft Auto V  36535  0.268428     26.842754
             Tom Clancy's Rainbow Six Siege  26134  0.511938     51.193847
                                       Rust  22901  0.439064     43.906380
                      ARK: Survival Evolved  20659  0.270633     27.063265
                              Rocket League  17890  0.501901     50.190050
                                   Terraria  17390  0.863140     86.313974
                                Garry's Mod  16723  0.778090     77.809006
                                   PAYDAY 2  16587  0.386809     38.680895
                               No Man's Sky  16063  0.198531     19.853079
                                   Among Us  15463  0.747914     74.791438
                                  Fallout 4  15231  0.282188     28.218764
                         


app_name:
  Chi2 statistic: 164298.0064
  p-value:        0.000000
  Degrees of freedom: 314
  Significant (p < 0.05): True


For the high cardinality categoricals, it is very clear that app_name is predictive. Within each class, there is distinct difference in their reccommendation rate, as well as having an extremely high chi-squared value, with 0.00 p-value, which shows statistical significance. Hence, we can easily conclude that app_name is important.

### Binary Bivariate Analysis
Binary features are treated as categorical, and thus as such, the same procedures will be followed.

In [25]:
# Binary
binary_cols = ["steam_purchase", "received_for_free", "written_during_early_access"]

sample_pd = df_sample.select(binary_cols + ["recommended"]).toPandas()

for c in binary_cols:
    # Recc rate
    bin_rec = df_sample.groupBy(c) \
        .agg(
            count("*").alias("total"),
            mean("recommended").alias("rec_rate")
        ) \
        .withColumn("rec_rate_pct", col("rec_rate") * 100) \
        .orderBy(c) \
        .toPandas()
    bin_rec[c] = bin_rec[c].map({0: "No", 1: "Yes"})
    print(f"\n=== {c} vs recommended ===")
    print(bin_rec.to_string(index=False))

    # Grouped bar chart
    bin_stack = df_sample.groupBy(c, "recommended") \
        .count() \
        .toPandas()
    bin_stack[c] = bin_stack[c].map({0: "No", 1: "Yes"})
    bin_stack["recommended"] = bin_stack["recommended"].map({0: "Not Recommended", 1: "Recommended"})

    fig = px.bar(
        bin_stack,
        x=c, y="count",
        color="recommended",
        title=f"{c} vs recommended",
        labels={"count": "Number of reviews", c: c},
        barmode="group",
        color_discrete_map={
            "Not Recommended": "#EF553B",
            "Recommended":     "#636EFA"
        },
        text="count"
    )
    fig.update_traces(texttemplate="%{text:,}", textposition="outside")
    show(fig, f"phase5_bivariate_binary_{c}")

    # Chi-Square test
    contingency = df_sample.groupBy(c, "recommended") \
        .count() \
        .toPandas() \
        .pivot(index=c, columns="recommended", values="count") \
        .fillna(0)

    chi2, pval, dof, _ = chi2_contingency(contingency)
    print(f"  Chi2: {chi2:.4f} | p-value: {pval:.6f} | Significant: {pval < 0.05}")

# One table summary
print("\n=== Recommendation rate summary ===")
summary_rows = []
for c in binary_cols:
    for val in [0, 1]:
        rec_rate = df_sample.filter(col(c) == val) \
            .agg(mean("recommended").alias("rec_rate")) \
            .collect()[0]["rec_rate"]
        summary_rows.append({
            "feature": c,
            "value":   "Yes" if val == 1 else "No",
            "rec_rate_pct": round(rec_rate * 100, 2)
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

fig = px.bar(
    summary_df,
    x="feature", y="rec_rate_pct",
    color="value",
    barmode="group",
    title="Recommendation rate by binary features (%)",
    labels={"rec_rate_pct": "Recommendation rate (%)", "feature": "Feature"},
    text="rec_rate_pct",
    color_discrete_map={
        "No":  "#EF553B",
        "Yes": "#636EFA"
    }
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
show(fig, "phase5_bivariate_binary_summary")


=== steam_purchase vs recommended ===
steam_purchase  total  rec_rate  rec_rate_pct
            No 166091  0.474186     47.418584
           Yes 530043  0.507400     50.740034


Plot saved → eda_plots/phase5_bivariate_binary_steam_purchase.html


  Chi2: 557.9265 | p-value: 0.000000 | Significant: True



=== received_for_free vs recommended ===
received_for_free  total  rec_rate  rec_rate_pct
               No 675869  0.499471     49.947105
              Yes  20265  0.499630     49.962990


Plot saved → eda_plots/phase5_bivariate_binary_received_for_free.html


  Chi2: 0.0014 | p-value: 0.970138 | Significant: False



=== written_during_early_access vs recommended ===
written_during_early_access  total  rec_rate  rec_rate_pct
                         No 618924  0.507379     50.737894
                        Yes  77210  0.436122     43.612226


Plot saved → eda_plots/phase5_bivariate_binary_written_during_early_access.html


  Chi2: 1393.9294 | p-value: 0.000000 | Significant: True

=== Recommendation rate summary ===


                    feature value  rec_rate_pct
             steam_purchase    No         47.42
             steam_purchase   Yes         50.74
          received_for_free    No         49.95
          received_for_free   Yes         49.96
written_during_early_access    No         50.74
written_during_early_access   Yes         43.61
Plot saved → eda_plots/phase5_bivariate_binary_summary.html


Looking at the binary features, we can see that recieved_for_free is actually shown to not be predictive. This is because of the low chi-squared value. Though there is a high p-value, making this finding not significant and thus we will still need to perform further investigations before we decide to drop it. As for the other features, they are performing as expected, and seem to be predictive.

## Correlation Matrix
We will create a correlation matrix to identify relationships between features. This is important as we want to remove highly correlated features from the dataset so as to prevent any possible bias.

In [26]:
# Assemble numeric columns into a vector
assembler = VectorAssembler(inputCols=num_cols, outputCol="features")
df_vector = assembler.transform(df_sample.select(num_cols).dropna())

# Compute Pearson correlation matrix
corr_matrix = Correlation.corr(df_vector, "features").collect()[0][0]
corr_array  = corr_matrix.toArray()

# Plot
fig = ff.create_annotated_heatmap(
    z=np.round(corr_array, 2),
    x=num_cols,
    y=num_cols,
    colorscale="RdBu",
    zmid=0,
    showscale=True,
    annotation_text=np.round(corr_array, 2).astype(str)
)
fig.update_layout(
    title="Correlation matrix — numeric features",
    xaxis_tickangle=-45,
    width=900,
    height=900
)
show(fig, "phase6_correlation_matrix")

# Print strong correlations
print("\n=== Strongly correlated pairs (|r| > 0.5) ===")
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        r = corr_array[i][j]
        if abs(r) > 0.5:
            print(f"  {num_cols[i]} × {num_cols[j]}: r = {r:.4f}")

Plot saved → eda_plots/phase6_correlation_matrix.html

=== Strongly correlated pairs (|r| > 0.5) ===
  votes_helpful × comment_count: r = 0.6084
  author_playtime_forever × author_playtime_at_review: r = 0.8559


Though we printed strong correlations as values above 0.5, this is for our own internal knowledge. In actual fact, it might not be a good idea to drop columns at that 0.5 threshold. Instead, we will first begin with the idea of keeping our threshold to 0.9, meaning that we do not have any super strong highly correlated values. It is important to note the interaction for author_playtime_forever × author_playtime_at_review: r = 0.8559, and that the value is quite close to the 0.9 threshold. We will take that into consideration during feature engineering, modelling, and evaluation and finetuning.

## Text Analysis
We will now do a deepdive into the review text themselves, identifying whether or not specific traits within the review text affect the target variable.

### Character and Word Count Analysis
We will use character and word count to uncover any potential trends.

In [27]:
# length columns
df_sample = df_sample \
    .withColumn("review_char_count", length(col("review"))) \
    .withColumn("review_word_count", size(split(col("review"), " ")))

# summaries
print("=== Review length summary stats ===")
df_sample.select(
    mean("review_char_count").alias("mean_chars"),
    stddev("review_char_count").alias("stddev_chars"),
    spark_min("review_char_count").alias("min_chars"),
    spark_max("review_char_count").alias("max_chars"),
    percentile_approx("review_char_count", 0.5).alias("median_chars"),
    mean("review_word_count").alias("mean_words"),
    stddev("review_word_count").alias("stddev_words"),
    spark_min("review_word_count").alias("min_words"),
    spark_max("review_word_count").alias("max_words"),
    percentile_approx("review_word_count", 0.5).alias("median_words")
).show(truncate=False)

# pull to pandas
length_pd = df_sample.select(
    "review_char_count", "review_word_count"
).toPandas()

# Character count distributions
fig = px.histogram(
    length_pd,
    x="review_char_count",
    nbins=100,
    title="Distribution of review character count",
    labels={"review_char_count": "Character count", "count": "Frequency"},
    marginal="box"
)
fig.update_layout(bargap=0.1)
show(fig, "phase8_review_char_dist")

# Word count distributions
fig = px.histogram(
    length_pd,
    x="review_word_count",
    nbins=100,
    title="Distribution of review word count",
    labels={"review_word_count": "Word count", "count": "Frequency"},
    marginal="box"
)
fig.update_layout(bargap=0.1)
show(fig, "phase8_review_word_dist")

=== Review length summary stats ===


+-----------------+-----------------+---------+---------+------------+-----------------+------------------+---------+---------+------------+
|mean_chars       |stddev_chars     |min_chars|max_chars|median_chars|mean_words       |stddev_words      |min_words|max_words|median_words|
+-----------------+-----------------+---------+---------+------------+-----------------+------------------+---------+---------+------------+
|302.7357893738849|644.8614574845615|1        |8000     |91          |54.76887926749735|115.01958100239301|1        |4094     |17          |
+-----------------+-----------------+---------+---------+------------+-----------------+------------------+---------+---------+------------+



Plot saved → eda_plots/phase8_review_char_dist.html
Plot saved → eda_plots/phase8_review_word_dist.html


### Count Comparison to Target
Here, we will use the data we obtained from above and plot them by reccommended to see any potential differences between classes.

In [28]:
# Review length mean by reccommended
print("=== Average review length by recommended ===")
df_sample.groupBy("recommended") \
    .agg(
        mean("review_char_count").alias("mean_chars"),
        mean("review_word_count").alias("mean_words"),
        percentile_approx("review_char_count", 0.5).alias("median_chars"),
        percentile_approx("review_word_count", 0.5).alias("median_words")
    ) \
    .orderBy("recommended") \
    .show(truncate=False)

# Pandas
length_rec_pd = df_sample.select(
    "review_char_count", "review_word_count", "recommended"
).toPandas()
length_rec_pd["recommended"] = length_rec_pd["recommended"].map({
    0: "Not Recommended", 1: "Recommended"
})

# Character count by reccommended boxplot
fig = px.box(
    length_rec_pd,
    x="recommended", y="review_char_count",
    title="Review character count by recommended",
    labels={"review_char_count": "Character count", "recommended": "Recommended"},
    color="recommended",
    color_discrete_map={
        "Not Recommended": "#EF553B",
        "Recommended":     "#636EFA"
    }
)
show(fig, "phase8_char_count_by_recommended")

# Word count by reccommended boxplot
fig = px.box(
    length_rec_pd,
    x="recommended", y="review_word_count",
    title="Review word count by recommended",
    labels={"review_word_count": "Word count", "recommended": "Recommended"},
    color="recommended",
    color_discrete_map={
        "Not Recommended": "#EF553B",
        "Recommended":     "#636EFA"
    }
)
show(fig, "phase8_word_count_by_recommended")

# Overlapping histogram
fig = px.histogram(
    length_rec_pd,
    x="review_word_count",
    color="recommended",
    nbins=100,
    barmode="overlay",
    opacity=0.6,
    title="Distribution of review word count by recommended",
    labels={"review_word_count": "Word count", "recommended": "Recommended"},
    color_discrete_map={
        "Not Recommended": "#EF553B",
        "Recommended":     "#636EFA"
    }
)
fig.update_layout(bargap=0.1)
show(fig, "phase8_word_count_dist_by_recommended")

=== Average review length by recommended ===


+-----------+------------------+-----------------+------------+------------+
|recommended|mean_chars        |mean_words       |median_chars|median_words|
+-----------+------------------+-----------------+------------+------------+
|0          |414.3901708224273 |75.01240414198466|155         |28          |
|1          |190.84698966356248|34.48285313285515|54          |10          |
+-----------+------------------+-----------------+------------+------------+



Plot saved → eda_plots/phase8_char_count_by_recommended.html
Plot saved → eda_plots/phase8_word_count_by_recommended.html
Plot saved → eda_plots/phase8_word_count_dist_by_recommended.html


## Export Cleaned CSV file
We will now export the cleaned CSV file that can be used for feature engineering and feature selection. This allows us to create a seperate ipynb file for that and segment our work better.

In [21]:
import glob
import shutil

# Write to temp
temp_path = "temp_csv_output"

df_sample.coalesce(1) \
  .write \
  .option("header", "true") \
  .option("quote", '"') \
  .option("escape", '"') \
  .mode("overwrite") \
  .csv(temp_path)

# Move and Rename
part_file = glob.glob(os.path.join(temp_path, "part-*.csv"))[0]
shutil.move(part_file, "cleaned_steam_reviews.csv")
shutil.rmtree(temp_path)

print("Saved to: cleaned_steam_reviews.csv")

Saved to: cleaned_steam_reviews.csv


In [22]:
pip install pandas
import pandas as pd

test = pd.read_csv("/Users/joshlim/Desktop/y4s2/bt4221/BT4221-Project-Steam-Reviews/cleaned_steam_reviews.csv")

test.size()

SyntaxError: invalid syntax (3153355656.py, line 1)